In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.pipeline import Pipeline

In [2]:
train_data = pd.read_csv(r'playground-series-s5e4\train.csv')
test_data = pd.read_csv(r'playground-series-s5e4\test.csv')
train_data

,id,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes
0,0,Mystery Matters,Episode 98,NaN,True Crime,74.81,Thursday,Night,NaN,0.0,Positive,31.41998
1,1,Joke Junction,Episode 26,119.80,Comedy,66.95,Saturday,Afternoon,75.95,2.0,Negative,88.01241
2,2,Study Sessions,Episode 16,73.90,Education,69.97,Tuesday,Evening,8.97,0.0,Negative,44.92531
3,3,Digital Digest,Episode 45,67.17,Technology,57.22,Monday,Morning,78.70,2.0,Positive,46.27824
4,4,Mind & Body,Episode 86,110.51,Health,80.07,Monday,Afternoon,58.68,3.0,Neutral,75.61031
...,...,...,...,...,...,...,...,...,...,...,...,...
749995,749995,Learning Lab,Episode 25,75.66,Education,69.36,Saturday,Morning,NaN,0.0,Negative,56.87058
749996,749996,Business Briefs,Episode 21,75.75,Business,35.21,Saturday,Night,NaN,2.0,Neutral,45.46242
749997,749997,Lifestyle Lounge,Episode 51,30.98,Lifestyle,78.58,Thursday,Morning,84.89,0.0,Negative,15.26000
749998,749998,Style Guide,Episode 47,108.98,Lifestyle,45.39,Thursday,Morning,93.27,0.0,Negative,100.72939


In [3]:
train_episode_no  = train_data['Episode_Title'].str.split().str[-1]
test_episode_no  = test_data['Episode_Title'].str.split().str[-1]

In [4]:
train_df = train_data.copy()
test_df = test_data.copy()

train_df['train_episode_no'] = train_episode_no
test_df['test_episode_no'] = test_episode_no
train_df['train_episode_no'] = train_df['train_episode_no'].astype(int)
test_df['test_episode_no'] = test_df['test_episode_no'].astype(int)
train_df.dtypes

id                               int64
Podcast_Name                    object
Episode_Title                   object
Episode_Length_minutes         float64
Genre                           object
Host_Popularity_percentage     float64
Publication_Day                 object
Publication_Time                object
Guest_Popularity_percentage    float64
Number_of_Ads                  float64
Episode_Sentiment               object
Listening_Time_minutes         float64
train_episode_no                 int64
dtype: object

In [5]:
train_df = train_df.drop('Episode_Title', axis = 1)
test_df = test_df.drop('Episode_Title', axis = 1)

In [6]:
train_df.drop_duplicates()
test_df.drop_duplicates()

,id,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,test_episode_no
0,750000,Educational Nuggets,78.96,Education,38.11,Saturday,Evening,53.33,1.0,Neutral,73
1,750001,Sound Waves,27.87,Music,71.29,Sunday,Morning,NaN,0.0,Neutral,23
2,750002,Joke Junction,69.10,Comedy,67.89,Friday,Evening,97.51,0.0,Positive,11
3,750003,Comedy Corner,115.39,Comedy,23.40,Sunday,Morning,51.75,2.0,Positive,73
4,750004,Life Lessons,72.32,Lifestyle,58.10,Wednesday,Morning,11.30,2.0,Neutral,50
...,...,...,...,...,...,...,...,...,...,...,...
249995,999995,Mind & Body,21.05,Health,65.77,Saturday,Evening,96.40,3.0,Negative,100
249996,999996,Joke Junction,85.50,Comedy,41.47,Saturday,Night,30.52,2.0,Negative,85
249997,999997,Joke Junction,12.11,Comedy,25.92,Thursday,Evening,73.69,1.0,Neutral,63
249998,999998,Market Masters,113.46,Business,43.47,Friday,Night,93.59,3.0,Positive,46


In [7]:
#train_df.shape
low_cat_cols = ['Genre', 'Publication_Day', 'Publication_Time', 'Episode_Sentiment']
high_cat_cols = ['Podcast_Name']
num_cols = ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Number_of_Ads']

input_df = train_df.drop('Listening_Time_minutes', axis = 1)
target = train_df['Listening_Time_minutes']
X_train, X_test, y_train, y_test = train_test_split(input_df, target, test_size = 0.2, random_state = 42)

In [8]:
train_df.columns

Index(['id', 'Podcast_Name', 'Episode_Length_minutes', 'Genre',
       'Host_Popularity_percentage', 'Publication_Day', 'Publication_Time',
       'Guest_Popularity_percentage', 'Number_of_Ads', 'Episode_Sentiment',
       'Listening_Time_minutes', 'train_episode_no'],
      dtype='object')

In [9]:
encoding = OneHotEncoder(handle_unknown = 'ignore')
ordinal_encoding = OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value = np.nan)
num_imputing = SimpleImputer(strategy = 'median')

In [10]:
column_transformer = ColumnTransformer(
    [
        ('one_hot_encoding', encoding, low_cat_cols),
        ('ordinal_encode', ordinal_encoding, high_cat_cols),
        ('num_cols', num_imputing, num_cols),
    ],
    remainder = 'drop',
    verbose_feature_names_out = True,
    sparse_threshold = 0
)

In [11]:
def root_mean_square_error(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return np.sqrt(mse)

In [12]:
cv = KFold(n_splits = 5, shuffle = True, random_state = 42)

rmse = make_scorer(root_mean_square_error, greater_is_better=False)
model = GradientBoostingRegressor(
    n_estimators= 100,
    learning_rate = 0.1,
    max_depth=4,
    min_samples_leaf=3,
    random_state = 42
)

model_pipeline = Pipeline([
    ('preprocessing', column_transformer),
    ('model', model)
])
cv_scorer = cross_validate(
    estimator = model_pipeline,
    X = X_train,
    y = y_train,
    scoring = rmse,
    cv = cv,
    return_estimator=True,
    return_train_score=True
)

In [13]:
mean_cv = np.mean(cv_scorer['test_score'])
std_cv = np.std(cv_scorer['test_score'])
fit_time = np.mean(cv_scorer['fit_time'])
score_time = np.mean(cv_scorer['score_time'])
print(f"{mean_cv:0.3f} ({std_cv:0.3f})",
f"fit: {fit_time:0.2f}",
f"secs pred: {score_time:0.2f} secs")

-13.154 (0.037) fit: 249.78 secs pred: 0.69 secs


In [20]:
test_predictions = np.zeros((len(X_test), cv.n_splits))
#test_predictions

In [21]:
#X_test.shape

In [16]:
for fold_idx, estimator in enumerate(cv_scorer['estimator']):
    test_predictions[:,fold_idx] = estimator.predict(X_test)

y_pred = np.mean(test_predictions, axis = 1)

In [19]:
#plt.scatter(y_test, y_pred)
#plt.show()

In [22]:
if 'y_test' in globals():
    test_rmse = root_mean_square_error(y_test, y_pred)
    print("Test RMSE:", test_rmse)

# Print predictions
#print("Final predictions:", y_pred)

Test RMSE: 13.102190529706792


In [29]:
final_predictions = np.zeros((len(test_df), cv.n_splits))

In [31]:
for fold_idx, estimator in enumerate(cv_scorer['estimator']):
    final_predictions[:,fold_idx] = estimator.predict(test_df)

y_final = np.mean(final_predictions, axis = 1)

In [32]:
y_final

array([56.33418363, 18.05999033, 49.80347057, ...,  7.16503069,
       74.13396914, 56.85424218], shape=(250000,))

In [33]:
final_df = pd.DataFrame()

In [35]:
final_df['id'] = test_df['id']
final_df['Listening_Time_minutes'] = y_final
final_df.head()

,id,Listening_Time_minutes
0,750000,56.334184
1,750001,18.059990
2,750002,49.803471
3,750003,80.097609
4,750004,48.971149


In [36]:
final_df.to_csv('listening_time_predictions.csv', index = False)